In [1]:
import os

# Must be set before importing mlflow — prevents hanging on telemetry fetch (MLflow 3.x)
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"

import mlflow
import mlflow.tracking
import tempfile
import uuid
import math
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datetime import datetime

# ── Connection ────────────────────────────────────────────────────────────────
# Use https:// — Cloudflare enforces a 308 redirect from http, which the
# MLflow Python client does NOT follow (hangs indefinitely).
# ─────────────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "https://mlflow-geoai.stelarea.com"

# 4-char unique suffix — keeps test runs in an isolated experiment,
# separate from the real "research-crop-mapping" experiment.
_UID = uuid.uuid4().hex[:4]
EXPERIMENT_NAME = f"research-crop-mapping-test-{_UID}"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"MLflow version  : {mlflow.__version__}")
print(f"Tracking URI    : {MLFLOW_TRACKING_URI}")
print(f"Test experiment : {EXPERIMENT_NAME}")

MLflow version  : 3.10.1
Tracking URI    : https://mlflow-geoai.stelarea.com
Test experiment : research-crop-mapping-test-7979


## 1. Test Connection

In [2]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    print(f"[INFO] Creating experiment '{EXPERIMENT_NAME}'")
    experiment_id = client.create_experiment(EXPERIMENT_NAME)
    experiment = client.get_experiment(experiment_id)

print(f"[OK] Connected")
print(f"Experiment    : {experiment.name}")
print(f"Experiment ID : {experiment.experiment_id}")
print(f"Artifact root : {experiment.artifact_location}")
# artifact_location should be mlflow-artifacts:/<id> (proxied) not s3://
assert experiment.artifact_location.startswith("mlflow-artifacts:"), \
    f"[WARN] Server not proxying artifacts — got: {experiment.artifact_location}"
print("[OK] Artifact proxy active (mlflow-artifacts:// URI)")

[INFO] Creating experiment 'research-crop-mapping-test-7979'
[OK] Connected
Experiment    : research-crop-mapping-test-7979
Experiment ID : 25
Artifact root : mlflow-artifacts:/25
[OK] Artifact proxy active (mlflow-artifacts:// URI)


## 2. Test `log_params`

In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"test_log_params_{RUN_TIMESTAMP}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({
        "model": "test-model",
        "encoder": "resnet18",
        "n_classes": 12,
        "batch_size": 16,
        "learning_rate": 1e-4,
        "n_epochs": 30,
        "patch_size": 256,
        "bands": "B2,B3,B4,B8",
    })
    print(f"[OK] log_params — run_id: {run.info.run_id}")

print(f"Run name : {run_name}")
print(f"Run ID   : {run.info.run_id}")

[OK] log_params — run_id: 901a125d1cbe4d1d9b920a3517df89c3
🏃 View run test_log_params_20260310-014847 at: https://mlflow-geoai.stelarea.com/#/experiments/25/runs/901a125d1cbe4d1d9b920a3517df89c3
🧪 View experiment at: https://mlflow-geoai.stelarea.com/#/experiments/25
Run name : test_log_params_20260310-014847
Run ID   : 901a125d1cbe4d1d9b920a3517df89c3


## 3. Test `log_metrics`

In [4]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"test_log_metrics_{RUN_TIMESTAMP}"
n_epochs = 10

with mlflow.start_run(run_name=run_name) as run:
    for epoch in range(1, n_epochs + 1):
        train_loss = 1.0 * math.exp(-0.2 * epoch) + 0.05
        val_loss   = 1.1 * math.exp(-0.18 * epoch) + 0.07
        miou       = 0.3 + 0.05 * epoch + 0.01 * (epoch % 3)
        pixel_acc  = 0.6 + 0.03 * epoch

        mlflow.log_metrics({
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "mIoU":       miou,
            "pixel_acc":  pixel_acc,
        }, step=epoch)

    mlflow.log_metrics({"best_mIoU": miou, "best_pixel_acc": pixel_acc})
    print(f"[OK] log_metrics — run_id: {run.info.run_id}")

print(f"Run name    : {run_name}")
print(f"Epochs logged: {n_epochs}")

[OK] log_metrics — run_id: 86a23f91eb4347eabc87cacf059b5a7f
🏃 View run test_log_metrics_20260310-014847 at: https://mlflow-geoai.stelarea.com/#/experiments/25/runs/86a23f91eb4347eabc87cacf059b5a7f
🧪 View experiment at: https://mlflow-geoai.stelarea.com/#/experiments/25
Run name    : test_log_metrics_20260310-014847
Epochs logged: 10


## 4. Test `log_artifact`

In [5]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"test_log_artifact_{RUN_TIMESTAMP}"

with mlflow.start_run(run_name=run_name) as run:
    with tempfile.TemporaryDirectory() as tmpdir:

        # --- Artifact 1: plain text ---
        txt_path = os.path.join(tmpdir, "test_summary.txt")
        with open(txt_path, "w") as f:
            f.write(f"MLflow artifact test\nRun: {run_name}\nTimestamp: {RUN_TIMESTAMP}\n")
        mlflow.log_artifact(txt_path)
        print(f"[OK] log_artifact (txt) : {os.path.basename(txt_path)}")

        # --- Artifact 2: JSON config ---
        cfg_path = os.path.join(tmpdir, "config.json")
        with open(cfg_path, "w") as f:
            json.dump({
                "model": "DeepLabV3+",
                "encoder": "resnet50",
                "n_classes": 12,
                "keep_classes": [3, 6, 24, 36, 37, 54, 61, 69, 75, 76, 220],
                "selected_bands": ["B2", "B3", "B4", "B8", "B11"],
            }, f, indent=2)
        mlflow.log_artifact(cfg_path)
        print(f"[OK] log_artifact (json): {os.path.basename(cfg_path)}")

        # --- Artifact 3: matplotlib figure ---
        fig, ax = plt.subplots(figsize=(6, 4))
        epochs_x = list(range(1, 11))
        ax.plot(epochs_x, [0.3 + 0.05 * e for e in epochs_x], marker="o", label="mIoU")
        ax.set_xlabel("Epoch"); ax.set_ylabel("mIoU")
        ax.set_title("Test: mIoU curve (simulated)"); ax.legend()
        fig.tight_layout()
        fig_path = os.path.join(tmpdir, "test_miou_curve.png")
        fig.savefig(fig_path, dpi=100); plt.close(fig)
        mlflow.log_artifact(fig_path)
        print(f"[OK] log_artifact (png) : {os.path.basename(fig_path)}")

    print(f"\nArtifact URI : {run.info.artifact_uri}")
    print(f"Run ID       : {run.info.run_id}")

[OK] log_artifact (txt) : test_summary.txt
[OK] log_artifact (json): config.json
[OK] log_artifact (png) : test_miou_curve.png

Artifact URI : mlflow-artifacts:/25/fde98c55bd1945fdaf07a2e1fa964842/artifacts
Run ID       : fde98c55bd1945fdaf07a2e1fa964842
🏃 View run test_log_artifact_20260310-014847 at: https://mlflow-geoai.stelarea.com/#/experiments/25/runs/fde98c55bd1945fdaf07a2e1fa964842
🧪 View experiment at: https://mlflow-geoai.stelarea.com/#/experiments/25


## 5. Verify — List Recent Test Runs

In [6]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=10,
)

print(f"Runs in '{EXPERIMENT_NAME}':\n")
for r in runs:
    name   = r.data.tags.get("mlflow.runName", "—")
    status = r.info.status
    print(f"  [{status}] {name}  (id: {r.info.run_id[:8]}...)")

Runs in 'research-crop-mapping-test-7979':

  [FINISHED] test_log_artifact_20260310-014847  (id: fde98c55...)
  [FINISHED] test_log_metrics_20260310-014847  (id: 86a23f91...)
  [FINISHED] test_log_params_20260310-014847  (id: 901a125d...)


## 6. Cleanup — Delete Test Runs (optional)

In [7]:
# Run this cell manually only when you want to remove test runs.
# Set DRY_RUN=False to actually delete.

DRY_RUN = True

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'test_%'",
    order_by=["start_time DESC"],
    max_results=50,
)

for r in runs:
    name = r.data.tags.get("mlflow.runName", "—")
    if DRY_RUN:
        print(f"[DRY RUN] would delete: {name}  ({r.info.run_id})")
    else:
        client.delete_run(r.info.run_id)
        print(f"[DELETED] {name}  ({r.info.run_id})")

if DRY_RUN:
    print("\nSet DRY_RUN=False to actually delete.")

[DRY RUN] would delete: test_log_artifact_20260310-014847  (fde98c55bd1945fdaf07a2e1fa964842)
[DRY RUN] would delete: test_log_metrics_20260310-014847  (86a23f91eb4347eabc87cacf059b5a7f)
[DRY RUN] would delete: test_log_params_20260310-014847  (901a125d1cbe4d1d9b920a3517df89c3)

Set DRY_RUN=False to actually delete.


# MLflow Tracking — Test Notebook

Tests connectivity, `log_params`, `log_metrics`, and `log_artifact` against the remote MLflow server.

**Server**: `https://mlflow-geoai.stelarea.com` (HTTPS required — Cloudflare enforces redirect)  
**Experiment**: `research-crop-mapping-test-<uid>` (4-char unique suffix per session, isolated from real experiments)

### Server configuration summary
| Setting | Value |
|---|---|
| MLflow version | 3.10.1 |
| Artifact storage | MinIO via `s3://research-geoai/mlflow/artifacts` |
| Artifact proxy | `--serve-artifacts` + `--artifacts-destination` (clients need no S3 creds) |
| CORS | `--cors-allowed-origins https://mlflow-geoai.stelarea.com` |

### Client requirement
`MLFLOW_DISABLE_TELEMETRY=true` must be set **before** importing mlflow — otherwise the client
hangs on a background telemetry fetch during `set_experiment()`.